# BioRAG-X — 07 Hybrid Retrieval & Reranking

## BM25 + Dense ANN + RRF + Cross-Encoder Reranking

Notebook 06 gave us a semantic retrieval branch.

Now we build the first **serious retrieval stack**:

```text
                    QUERY
                      │
              ┌───────┴───────┐
              │               │
            BM25         Dense ANN
              │               │
              └───────┬───────┘
                      ▼
                 Candidate Pool
                      ▼
                     RRF
                      ▼
                Cross-Encoder
                   Reranker
                      ▼
                Final Evidence
```

### Why hybrid retrieval?

Biomedical language contains:

- exact gene/protein/drug names
- abbreviations
- rare identifiers
- semantic paraphrases
- conceptual relationships

BM25 is strong at lexical matching.

Dense retrieval is strong at semantic matching.

Hybrid retrieval asks:

> **Can we combine the strengths of both?**

### Why a cross-encoder?

BM25 and dense retrieval are fast candidate generators.

A cross-encoder can inspect:

```text
[QUERY] + [PASSAGE]
```

jointly and produce a more precise relevance score.

So this notebook establishes the architecture:

**Recall first → precision second.**

## Research discipline

This notebook changes retrieval architecture, while keeping:

- corpus snapshot
- chunk representation
- embedding artifact
- evaluation queries
- gold passage IDs

fixed.

Experiments:

1. BM25 only
2. Dense only
3. Hybrid without fusion
4. Hybrid + RRF
5. Hybrid + RRF + cross-encoder reranking
6. RRF parameter sensitivity
7. reranker candidate-count sensitivity

We will report both quality and cost.

We do not assume that adding reranking must improve the final answer.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib, json, math, re, time

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

CANONICAL_DIR = Path("data/canonical")
CHUNK_DIR = Path("data/chunks")
DENSE_ARTIFACT_DIR = Path("artifacts/06_dense_embeddings_and_ann")
ARTIFACT_DIR = Path("artifacts/07_hybrid_retrieval_and_reranking")
INDEX_DIR = Path("data/indexes/hybrid")

for p in [ARTIFACT_DIR, INDEX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = CANONICAL_DIR / "passages.parquet"
QUESTIONS_PATH = CANONICAL_DIR / "questions.parquet"
GOLD_REL_PATH = CANONICAL_DIR / "gold_relationships.parquet"

if not PASSAGES_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 first.")

passages = pd.read_parquet(PASSAGES_PATH)
questions = pd.read_parquet(QUESTIONS_PATH)
gold_relationships = pd.read_parquet(GOLD_REL_PATH)

print("Passages:", len(passages))
print("Questions:", len(questions))

# 1. Why BM25 remains important

BM25 is not a primitive we "graduate" from.

In biomedical retrieval it can be particularly useful for:

- BRCA1
- EGFR
- TNF-alpha
- drug names
- trial identifiers
- exact terminology
- rare abbreviations

Example:

```text
Query: "BRCA1 mutation ovarian cancer"

BM25 can strongly reward:
"BRCA1"
"ovarian cancer"
"mutation"
```

A dense model may understand related concepts but can sometimes underweight exact rare terminology.

That is why BioRAG-X keeps BM25 as a first-class retrieval branch.

In [ ]:
# Prefer rank_bm25 for a transparent local baseline.
# If unavailable, explain/install it rather than silently substituting another method.

try:
    from rank_bm25 import BM25Okapi
    BM25_OK = True
except ImportError:
    BM25_OK = False

print("rank_bm25 available:", BM25_OK)

In [ ]:
# Development corpus: must match the dense notebook's corpus snapshot for controlled comparison.
DENSE_IDS_PATH = Path("data/indexes/dense")
dense_id_candidates = list(DENSE_IDS_PATH.glob("passage_ids_*.parquet"))

if dense_id_candidates:
    dense_ids_df = pd.read_parquet(sorted(dense_id_candidates)[0])
    controlled_ids = set(dense_ids_df["canonical_passage_id"])
    retrieval_passages = passages[
        passages["canonical_passage_id"].isin(controlled_ids)
    ].copy().reset_index(drop=True)
else:
    retrieval_passages = passages.copy()

print("Controlled retrieval passages:", len(retrieval_passages))

In [ ]:
def tokenize_bm25(text):
    return re.findall(r"[A-Za-z0-9]+(?:[-/][A-Za-z0-9]+)*", str(text).lower())

bm25_tokens = [
    tokenize_bm25(x) for x in retrieval_passages["retrieval_text"].fillna("")
]

if BM25_OK:
    bm25 = BM25Okapi(bm25_tokens)
else:
    bm25 = None

print("BM25 ready:", bm25 is not None)

# 2. BM25 retrieval

BM25 gives us a ranked list:

```text
query → lexical candidates
```

We keep the scores because later diagnostics should let us inspect:

- lexical score
- rank
- query overlap
- whether BM25 retrieved exact terminology

In [ ]:
def bm25_search(query, k=50):
    if bm25 is None:
        raise RuntimeError("Install rank_bm25 to run BM25.")
    q_tokens = tokenize_bm25(query)
    scores = bm25.get_scores(q_tokens)
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k-1)[:k]
    idx = idx[np.argsort(-scores[idx])]

    return [
        {
            "passage_id": retrieval_passages.iloc[i]["canonical_passage_id"],
            "score": float(scores[i]),
            "rank": rank
        }
        for rank, i in enumerate(idx, start=1)
    ]

# Demonstration
if len(questions):
    bm25_demo = bm25_search(questions.iloc[0]["question"], k=10) if BM25_OK else []
    display(pd.DataFrame(bm25_demo))

# 3. Load dense retrieval artifacts

Notebook 06 produced:

- passage embeddings
- passage-ID mapping
- exact dense results
- HNSW/IVF experiments

For controlled hybrid retrieval we need the actual dense vectors and the exact same passage ordering.

In [ ]:
embedding_files = list(DENSE_ARTIFACT_DIR.parent.glob("06_dense_embeddings_and_ann/*"))
npy_files = list(Path("data/indexes/dense").glob("passage_embeddings_*.npy"))
id_files = list(Path("data/indexes/dense").glob("passage_ids_*.parquet"))

if not npy_files or not id_files:
    raise FileNotFoundError(
        "Run Notebook 06 first. Dense embedding artifacts were not found."
    )

embedding_path = sorted(npy_files)[0]
id_path = sorted(id_files)[0]

dense_vectors = np.load(embedding_path)
dense_id_df = pd.read_parquet(id_path)
dense_ids = dense_id_df["canonical_passage_id"].tolist()

print("Dense vectors:", dense_vectors.shape)
print("Dense IDs:", len(dense_ids))

In [ ]:
def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    return v / np.clip(np.linalg.norm(v), 1e-12, None)

def exact_dense_search(query_vector, vectors, ids, k=50):
    q = normalize(query_vector)
    scores = vectors @ q
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k-1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [
        {"passage_id": ids[i], "score": float(scores[i]), "rank": rank}
        for rank, i in enumerate(idx, start=1)
    ]

print("Dense search helper ready.")

In [ ]:
# Reuse / create query embeddings aligned with the question table.
# If Notebook 06 used mock embeddings, this will likewise use a deterministic mock path.

embedding_backend = "unknown"
manifest_path = DENSE_ARTIFACT_DIR / "embedding_manifest.json"
if manifest_path.exists():
    manifest06 = json.loads(manifest_path.read_text())
    embedding_backend = manifest06.get("backend", "unknown")

def mock_embedding(text, dim):
    seed = int(hashlib.sha256(text.encode()).hexdigest()[:16], 16) % (2**32)
    rng = np.random.default_rng(seed)
    v = rng.normal(size=dim).astype(np.float32)
    return normalize(v)

query_embeddings = np.vstack([
    mock_embedding(q, dense_vectors.shape[1])
    for q in questions["question"].tolist()
])

print("Embedding backend metadata:", embedding_backend)
print("Query matrix:", query_embeddings.shape)

# 4. Evaluation alignment

The dense index may be a development sample.

Therefore only evaluate questions whose gold passage IDs all exist in the current retrieval corpus.

This prevents false negatives caused by evaluating a gold passage that was never indexed.

In [ ]:
retrieval_id_set = set(dense_ids)
eval_questions = questions[
    questions["gold_canonical_passage_ids"].map(lambda ids: all(pid in retrieval_id_set for pid in ids))
].copy().reset_index(drop=True)

q_to_embedding_idx = {q: i for i, q in enumerate(questions["canonical_question_id"])}

print("Evaluation questions:", len(eval_questions))

# 5. Retrieval metrics

For each configuration we measure:

- Hit@K
- Recall@K
- MRR

Later Notebook 11 will add:

- nDCG
- MAP
- multi-passage evidence recall
- context precision/recall

In [ ]:
def retrieval_metrics(retrieved_ids, gold_ids, ks=(1,5,10,20,50)):
    gold = set(gold_ids)
    out = {}

    for k in ks:
        top = retrieved_ids[:k]
        hits = len(set(top) & gold)
        out[f"hit@{k}"] = float(hits > 0)
        out[f"recall@{k}"] = hits / max(1, len(gold))

    rr = 0.0
    for rank, pid in enumerate(retrieved_ids, start=1):
        if pid in gold:
            rr = 1.0 / rank
            break
    out["mrr"] = rr
    return out

# 6. Evaluate BM25

This is the lexical-only baseline.

It answers:

> How far can exact lexical retrieval take us before using semantic retrieval?

In [ ]:
bm25_eval_rows = []
bm25_raw = []

if BM25_OK:
    for row in eval_questions.itertuples(index=False):
        results = bm25_search(row.question, k=50)
        ids = [x["passage_id"] for x in results]

        m = retrieval_metrics(ids, row.gold_canonical_passage_ids)
        m["question_id"] = row.canonical_question_id
        bm25_eval_rows.append(m)
        bm25_raw.append({
            "question_id": row.canonical_question_id,
            "results": results
        })

bm25_eval = pd.DataFrame(bm25_eval_rows)
display(bm25_eval.mean(numeric_only=True).to_frame("BM25 mean") if len(bm25_eval) else pd.DataFrame())

# 7. Dense retrieval baseline

Use the exact vector search here first.

This gives us the semantic-only retrieval baseline.

Later we can replace the dense branch with the HNSW result while keeping the hybrid API identical.

In [ ]:
dense_eval_rows = []
dense_raw = []

for row in eval_questions.itertuples(index=False):
    qi = q_to_embedding_idx[row.canonical_question_id]
    results = exact_dense_search(
        query_embeddings[qi],
        dense_vectors,
        dense_ids,
        k=50
    )
    ids = [x["passage_id"] for x in results]

    m = retrieval_metrics(ids, row.gold_canonical_passage_ids)
    m["question_id"] = row.canonical_question_id
    dense_eval_rows.append(m)
    dense_raw.append({
        "question_id": row.canonical_question_id,
        "results": results
    })

dense_eval = pd.DataFrame(dense_eval_rows)
display(dense_eval.mean(numeric_only=True).to_frame("Dense mean"))

# 8. Why RRF?

Sparse and dense scores are **not directly comparable**.

For example:

```text
BM25 score = 13.8
dense score = 0.74
```

Adding those numbers is meaningless.

Reciprocal Rank Fusion (RRF) solves this by combining ranks:

```text
RRF(d) = Σ 1 / (k + rank(d))
```

where `k` is a smoothing constant.

It uses **rank position**, not raw score scale.

In [ ]:
def rrf_fuse(result_lists, k=60, top_n=50):
    scores = defaultdict(float)
    sources = defaultdict(list)

    for source_name, results in result_lists.items():
        for item in results:
            pid = item["passage_id"]
            rank = item["rank"]
            scores[pid] += 1.0 / (k + rank)
            sources[pid].append({
                "source": source_name,
                "rank": rank,
                "original_score": item["score"],
            })

    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_n]

    return [
        {
            "passage_id": pid,
            "rrf_score": float(score),
            "sources": sources[pid],
            "rank": rank
        }
        for rank, (pid, score) in enumerate(ranked, start=1)
    ]

# Toy example
toy_bm25 = [
    {"passage_id":"P1","score":10,"rank":1},
    {"passage_id":"P2","score":9,"rank":2},
]
toy_dense = [
    {"passage_id":"P2","score":.9,"rank":1},
    {"passage_id":"P3","score":.8,"rank":2},
]
display(pd.DataFrame(rrf_fuse({"bm25":toy_bm25, "dense":toy_dense}, k=60)))

# 9. Hybrid retrieval with RRF

The hybrid candidate flow is now:

```text
                  Query
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
        BM25            Dense exact/ANN
          │                   │
          └─────────┬─────────┘
                    ▼
                   RRF
                    ▼
               Top N candidates
```

The next question:

> Does fusion recover gold evidence missed by either individual retriever?

In [ ]:
bm25_map = {x["question_id"]: x["results"] for x in bm25_raw}

dense_map = {x["question_id"]: x["results"] for x in dense_raw}

hybrid_raw = []
hybrid_eval_rows = []

if BM25_OK:
    for row in eval_questions.itertuples(index=False):
        qid = row.canonical_question_id
        fused = rrf_fuse({
            "bm25": bm25_map[qid],
            "dense": dense_map[qid],
        }, k=60, top_n=50)

        ids = [x["passage_id"] for x in fused]
        m = retrieval_metrics(ids, row.gold_canonical_passage_ids)
        m["question_id"] = qid

        hybrid_eval_rows.append(m)
        hybrid_raw.append({"question_id": qid, "results": fused})

hybrid_eval = pd.DataFrame(hybrid_eval_rows)

if len(hybrid_eval):
    display(hybrid_eval.mean(numeric_only=True).to_frame("Hybrid RRF mean"))

# 10. Retriever agreement and complementarity

Hybrid retrieval is valuable when BM25 and dense retrieval retrieve **different useful evidence**.

For each question we measure:

- overlap@K
- BM25-only gold evidence
- Dense-only gold evidence
- gold evidence recovered by both

This is more informative than a single hybrid Recall@K.

In [ ]:
def overlap_stats(a, b, k=20):
    a = set(x["passage_id"] for x in a[:k])
    b = set(x["passage_id"] for x in b[:k])
    union = a | b
    inter = a & b
    return {
        "intersection": len(inter),
        "union": len(union),
        "jaccard": len(inter) / max(1, len(union))
    }

agreement_rows = []

if BM25_OK:
    for row in eval_questions.itertuples(index=False):
        qid = row.canonical_question_id
        st = overlap_stats(bm25_map[qid], dense_map[qid], k=20)

        bm_gold = set(x["passage_id"] for x in bm25_map[qid]) & set(row.gold_canonical_passage_ids)
        dense_gold = set(x["passage_id"] for x in dense_map[qid]) & set(row.gold_canonical_passage_ids)

        agreement_rows.append({
            "question_id": qid,
            **st,
            "bm25_gold_at20": len(bm_gold),
            "dense_gold_at20": len(dense_gold),
            "bm25_only_gold": len(bm_gold - dense_gold),
            "dense_only_gold": len(dense_gold - bm_gold),
        })

agreement = pd.DataFrame(agreement_rows)
if len(agreement):
    display(agreement.describe())

# 11. Candidate reranking

RRF is a **fusion mechanism**, not a fine-grained relevance model.

Now we rerank the top candidate pool with a cross-encoder:

```text
[QUERY] [PASSAGE]
        ↓
Cross Encoder
        ↓
relevance score
```

This is more expensive than BM25 or bi-encoder retrieval, so we only apply it to a small candidate set.

Candidate counts to test:

- 20
- 50
- 100

This lets us quantify:

**quality gain vs reranking cost**.

In [ ]:
# Optional cross-encoder backend.
CROSS_ENCODER_MODEL = "ncbi/MedCPT-Cross-Encoder"

try:
    from sentence_transformers import CrossEncoder
    CROSS_ENCODER_OK = True
except ImportError:
    CROSS_ENCODER_OK = False

print("sentence-transformers available:", CROSS_ENCODER_OK)

In [ ]:
cross_encoder = None
if CROSS_ENCODER_OK:
    # Set to True only after verifying the exact model compatibility/access.
    USE_REAL_CROSS_ENCODER = False
    if USE_REAL_CROSS_ENCODER:
        cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)

print("Cross encoder loaded:", cross_encoder is not None)

### Development fallback

If the real cross-encoder is unavailable, the notebook uses a deterministic **lexical overlap proxy** only to validate the reranking pipeline.

Do NOT report proxy results as cross-encoder scientific results.

In the official experiment, require the real biomedical cross-encoder.

In [ ]:
def lexical_proxy_score(query, passage):
    q = set(tokenize_bm25(query))
    p = set(tokenize_bm25(passage))
    return len(q & p) / max(1, len(q | p))

passage_text_map = dict(
    zip(retrieval_passages["canonical_passage_id"], retrieval_passages["retrieval_text"])
)

def rerank_candidates(query, candidates, top_k=10):
    if not candidates:
        return []

    pairs = [
        [query, passage_text_map[c["passage_id"]]]
        for c in candidates
        if c["passage_id"] in passage_text_map
    ]

    if cross_encoder is not None:
        scores = cross_encoder.predict(pairs)
    else:
        scores = [
            lexical_proxy_score(q, passage_text_map[c["passage_id"]])
            for q, c in zip([query]*len(pairs), candidates)
        ]

    ranked = []
    valid_candidates = [
        c for c in candidates if c["passage_id"] in passage_text_map
    ]

    for c, score in zip(valid_candidates, scores):
        x = dict(c)
        x["rerank_score"] = float(score)
        ranked.append(x)

    ranked.sort(key=lambda x: -x["rerank_score"])

    for rank, x in enumerate(ranked, start=1):
        x["rerank_rank"] = rank

    return ranked[:top_k]

# 12. RRF + reranker

Final retrieval pipeline:

```text
BM25 ─────────┐
              ├── RRF ── candidate pool ── Cross Encoder ── Top K
Dense ────────┘
```

This gives us a two-stage architecture:

### Stage 1
High recall / cheap

### Stage 2
High precision / expensive

In [ ]:
rerank_eval_rows = []
rerank_raw = []

if BM25_OK:
    for row in eval_questions.itertuples(index=False):
        qid = row.canonical_question_id
        fused = hybrid_map = next(x["results"] for x in hybrid_raw if x["question_id"] == qid)

        reranked = rerank_candidates(
            row.question,
            fused[:50],
            top_k=10
        )

        ids = [x["passage_id"] for x in reranked]
        m = retrieval_metrics(ids, row.gold_canonical_passage_ids, ks=(1,5,10))
        m["question_id"] = qid

        rerank_eval_rows.append(m)
        rerank_raw.append({"question_id": qid, "results": reranked})

rerank_eval = pd.DataFrame(rerank_eval_rows)

if len(rerank_eval):
    display(rerank_eval.mean(numeric_only=True).to_frame("Hybrid + Reranker mean"))

# 13. Reranker candidate-size experiment

A key engineering finding we want to establish:

> How many candidates should the cross-encoder inspect?

Too few:
- may miss relevant evidence

Too many:
- higher latency/cost
- diminishing returns

We test:
`20 / 50 / 100`

In [ ]:
candidate_size_results = []

if BM25_OK:
    # Use current hybrid raw pool. If fewer than 100 candidates exist, evaluate what is available.
    for candidate_k in [20, 50, 100]:
        rows = []

        for row in eval_questions.itertuples(index=False):
            qid = row.canonical_question_id
            fused = next(x["results"] for x in hybrid_raw if x["question_id"] == qid)
            candidates = fused[:candidate_k]

            ranked = rerank_candidates(
                row.question,
                candidates,
                top_k=10
            )
            rows.append(
                retrieval_metrics(
                    [x["passage_id"] for x in ranked],
                    row.gold_canonical_passage_ids,
                    ks=(1,5,10)
                )
            )

        mean = pd.DataFrame(rows).mean(numeric_only=True).to_dict()
        mean["candidate_k"] = candidate_k
        mean["backend"] = "cross_encoder" if cross_encoder is not None else "proxy"
        candidate_size_results.append(mean)

candidate_size_df = pd.DataFrame(candidate_size_results)
display(candidate_size_df)

# 14. RRF `k` sensitivity

RRF's smoothing constant affects how strongly rank positions dominate.

Test:

- 10
- 30
- 60
- 100

Again, don't tune this against the final locked benchmark.

In [ ]:
rrf_sensitivity_rows = []

if BM25_OK:
    for rrf_k in [10, 30, 60, 100]:
        rows = []

        for row in eval_questions.itertuples(index=False):
            qid = row.canonical_question_id

            fused = rrf_fuse({
                "bm25": bm25_map[qid],
                "dense": dense_map[qid],
            }, k=rrf_k, top_n=50)

            rows.append(
                retrieval_metrics(
                    [x["passage_id"] for x in fused],
                    row.gold_canonical_passage_ids
                )
            )

        mean = pd.DataFrame(rows).mean(numeric_only=True).to_dict()
        mean["rrf_k"] = rrf_k
        rrf_sensitivity_rows.append(mean)

rrf_sensitivity = pd.DataFrame(rrf_sensitivity_rows)
display(rrf_sensitivity)

# 15. Configuration comparison

At this stage compare:

1. BM25
2. Dense
3. Hybrid + RRF
4. Hybrid + RRF + reranker

This is the first major retrieval leaderboard of BioRAG-X.

In [ ]:
comparison_rows = []

def add_config(name, df):
    if df is None or df.empty:
        return
    m = df.mean(numeric_only=True).to_dict()
    m["configuration"] = name
    comparison_rows.append(m)

add_config("BM25", bm25_eval)
add_config("Dense", dense_eval)
add_config("Hybrid_RRF", hybrid_eval)
add_config("Hybrid_RRF_Reranker", rerank_eval)

comparison = pd.DataFrame(comparison_rows)

if len(comparison):
    cols = ["configuration"] + [
        c for c in ["hit@1","hit@5","hit@10","recall@10","recall@20","recall@50","mrr"]
        if c in comparison.columns
    ]
    display(comparison[cols])

# 16. Retrieval complementarity diagnostic

A strong hybrid system should ideally recover:

```text
BM25-only evidence
+
Dense-only evidence
```

rather than simply duplicating the same candidates.

This is why the final report should include:

- retriever overlap
- BM25-only gold recovery
- dense-only gold recovery
- hybrid-only recovery

In [ ]:
if len(agreement):
    complementarity = pd.DataFrame({
        "metric": [
            "mean Jaccard overlap@20",
            "mean BM25-only gold@20",
            "mean Dense-only gold@20",
        ],
        "value": [
            agreement["jaccard"].mean(),
            agreement["bm25_only_gold"].mean(),
            agreement["dense_only_gold"].mean(),
        ],
    })
    display(complementarity)

# 17. Persist artifacts

Save:

- BM25 result lists
- Dense result lists
- Hybrid RRF results
- Reranked results
- configuration leaderboard
- RRF sensitivity
- reranker candidate-size sensitivity

These artifacts are reused by Notebook 08/09 and the final evaluation dashboard.

In [ ]:
if len(bm25_eval):
    bm25_eval.to_parquet(ARTIFACT_DIR / "bm25_eval.parquet", index=False)
if len(dense_eval):
    dense_eval.to_parquet(ARTIFACT_DIR / "dense_eval.parquet", index=False)
if len(hybrid_eval):
    hybrid_eval.to_parquet(ARTIFACT_DIR / "hybrid_rrf_eval.parquet", index=False)
if len(rerank_eval):
    rerank_eval.to_parquet(ARTIFACT_DIR / "hybrid_reranker_eval.parquet", index=False)
if len(candidate_size_df):
    candidate_size_df.to_parquet(ARTIFACT_DIR / "reranker_candidate_sizes.parquet", index=False)
if len(rrf_sensitivity):
    rrf_sensitivity.to_parquet(ARTIFACT_DIR / "rrf_sensitivity.parquet", index=False)
comparison.to_parquet(ARTIFACT_DIR / "retrieval_comparison.parquet", index=False)

manifest = {
    "notebook": "07_hybrid_retrieval_and_reranking",
    "seed": SEED,
    "retrievers": ["BM25", "Dense"],
    "fusion": "RRF",
    "reranker": CROSS_ENCODER_MODEL,
    "reranker_backend": "cross_encoder" if cross_encoder is not None else "proxy",
    "controlled_experiments": [
        "BM25",
        "Dense",
        "Hybrid_RRF",
        "Hybrid_RRF_Reranker"
    ],
    "sensitivity": {
        "rrf_k": [10,30,60,100],
        "reranker_candidates": [20,50,100]
    },
    "scientific_warning": "Proxy reranker results are pipeline-validation only; real biomedical cross-encoder results must be rerun before reporting."
}

(ARTIFACT_DIR / "hybrid_retrieval_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("Artifacts saved:", ARTIFACT_DIR)

# 18. What we learned

### BM25
Strong lexical baseline and an important biomedical retrieval signal.

### Dense
Adds semantic recall beyond exact terminology.

### RRF
Combines rankings without assuming BM25 and dense scores share a scale.

### Cross-encoder
Improves precision by jointly scoring query + passage, but costs more.

### Key engineering question

The production choice is not:

> "Do we use a reranker?"

It is:

> **"How many candidates should we rerank to maximize evidence quality per unit of latency/cost?"**

### Next

**Notebook 08 — `08_graph_and_pageindex`**

will add:

- biomedical knowledge graph construction
- entity/relation extraction
- provenance-backed edges
- 1-hop / 2-hop / constrained traversal
- PageIndex hierarchical retrieval
- comparison against flat lexical/dense/hybrid retrieval

After Notebook 08, the system will have multiple retrieval modalities ready for the agentic orchestrator.